# 🔌 01: LLM은 인터페이스다

---

| 항목 | 내용 |
|------|------|
| **목표** | LLM이 '말 잘하는 챗봇'이 아니라 '구조화된 입출력 인터페이스'임을 보여준다 |
| **예상 실행 시간** | ⏱️ 빠른 시연 3분 / 전체 시연 10분 |
| **API 키** | ✅ 권장 (없으면 mock 응답으로 대체) |
| **이전 노트북과의 연결** | 00에서 환경을 준비했다. 이제 LLM의 역할을 제대로 보자. |

---

## 🎯 오늘 볼 것

**같은 입력에 대해 3가지 패턴 비교:**

```
입력: 회의 메모 텍스트

패턴 A: 자유 생성     → 요약해줘
패턴 B: 구조화 출력   → JSON으로 TODO 추출
패턴 C: 함수 인자 파싱 → 자연어를 API 파라미터로 변환
```

**핵심 메시지:**  
> LLM의 가치는 '말 잘하기'가 아닙니다.  
> **구조화, 제약, 형식화**가 핵심입니다.  
> LLM은 AI 시스템의 공통 인터페이스가 됐습니다.

In [ ]:
# 필요한 경우 설치
!pip install -q openai

import json
import os
from IPython.display import display, HTML
print("✅ 임포트 완료")

## 1️⃣ 환경 설정

In [ ]:
import os, json
from IPython.display import display, HTML

# ============================================================
# ✏️ API 키 설정 (없으면 mock 모드)
# ============================================================
# .env 에서 API 키 로드 (python-dotenv 필요, Colab 에서는 직접 입력 가능)
try:
    from dotenv import load_dotenv
    from pathlib import Path
    for _p in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]:
        if (_p / ".env").exists():
            load_dotenv(_p / ".env"); break
except ImportError:
    pass

OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "")

def setup(api_key=""):
    key = api_key or os.environ.get("OPENAI_API_KEY", "")
    if key and key not in ("", "sk-...", "your-api-key-here"):
        try:
            from openai import OpenAI
            c = OpenAI(api_key=key)
            print("✅ API 모드")
            return "api", c
        except:
            pass
    print("💡 로컬(Mock) 모드 - API 키 없이도 데모 진행 가능")
    return "local", None

MODE, client = setup(OPENAI_API_KEY)

# ──────────────────────────────────────────────
# LLM 호출 헬퍼 (fallback 포함)
# ──────────────────────────────────────────────
def call_llm(prompt, system="당신은 도움이 되는 어시스턴트입니다.",
             mock=None, temperature=0.7):
    if MODE == "api" and client:
        try:
            r = client.chat.completions.create(
                model="gpt-4o-mini",
                messages=[{"role":"system","content":system},
                           {"role":"user","content":prompt}],
                temperature=temperature, max_tokens=800
            )
            return r.choices[0].message.content.strip()
        except Exception as e:
            print(f"⚠️ API 실패: {e}")
    return f"[Mock 응답]\n{mock}" if mock else "[로컬 모드]"

def call_llm_json(prompt, system="JSON만 출력하세요.", mock=None):
    if MODE == "api" and client:
        try:
            r = client.chat.completions.create(
                model="gpt-4o-mini",
                messages=[{"role":"system","content":system},
                           {"role":"user","content":prompt}],
                response_format={"type":"json_object"},
                temperature=0.1, max_tokens=1000
            )
            return json.loads(r.choices[0].message.content)
        except Exception as e:
            print(f"⚠️ API 실패: {e}")
    return mock if mock else {"error": "로컬 모드"}

print(f"모드: {MODE}")

## 2️⃣ 예제 데이터 - 업무형 회의 메모

In [ ]:
import sys
from pathlib import Path


def _ensure_project_root_on_path() -> None:
    cwd = Path.cwd().resolve()
    candidates = [cwd, *cwd.parents]
    try:
        candidates.extend(path for path in cwd.iterdir() if path.is_dir())
    except OSError:
        pass

    for candidate in candidates:
        if (candidate / "helpers" / "sample_data.py").exists():
            candidate_str = str(candidate)
            if candidate_str not in sys.path:
                sys.path.insert(0, candidate_str)
            return

    raise ModuleNotFoundError(
        "프로젝트 루트를 찾지 못했습니다. `ai_special_course` 저장소를 clone 한 뒤 "
        "repo root 또는 notebooks 디렉터리에서 노트북을 실행하세요."
    )


_ensure_project_root_on_path()

# 강의용 샘플 데이터
# 실제 업무에서 자주 보이는 형태로 작성
from helpers.sample_data import MEETING_MEMO, SUPPORT_TICKET_TEXT, USER_SEARCH_REQUEST
# Colab 사용 시: !git clone <repo> 후 sys.path 에 추가하거나, sample_data.py 를 /content 에 업로드하세요.

print("✅ 예제 데이터 준비 완료")
print("  1. 회의 메모 (스프린트 회의)")
print("  2. 고객 문의 텍스트 (Support Ticket)")
print("  3. 자연어 검색 요청 (Search Intent)")

---

## 3️⃣ 패턴 A: 자유 생성 (일반 요약)

가장 기본적인 사용법. 하지만 **구조가 없어서 후처리가 어렵습니다.**

In [ ]:
# ─────────────────────────────────────────
# 패턴 A: 자유로운 텍스트 요약
# ─────────────────────────────────────────
prompt_free = f"""아래 회의 메모를 간단히 요약해줘.

{MEETING_MEMO}"""

mock_free = """이번 스프린트 회의에서 CloudSync v2.3 배포가 11월 26일로 확정됐습니다.
이수진이 마이그레이션 스크립트를 금요일까지 검토해야 하고,
박민재의 로딩 버그는 오늘 중으로 처리 예정입니다.
QA에서 3개 이슈가 발견됐으며, 다음 스프린트 플래닝은 12월 2일입니다."""

result_a = call_llm(prompt_free, mock=mock_free)

print("=" * 55)
print("  패턴 A: 자유 생성 (일반 요약)")
print("=" * 55)
print(result_a)
print()
print("⚠️  문제점: 이 결과를 코드로 처리하기 어렵습니다.")
print("   담당자 추출, 마감일 추출, TODO 분리가 모두 불확실합니다.")

## 4️⃣ 패턴 B: 구조화 출력 (JSON)

**같은 입력**이지만 출력 형식을 지정하면 완전히 다른 결과가 나옵니다.  
이 구조화된 결과는 **다음 시스템으로 넘기거나 DB에 저장**할 수 있습니다.

In [ ]:
import json, pandas as pd

# ─────────────────────────────────────────
# 패턴 B: JSON 구조화 출력
# ─────────────────────────────────────────
prompt_json = f"""아래 회의 메모에서 정보를 추출해서 JSON으로 반환하세요.

반환 형식:
{{
  "summary": "회의 요약 (1-2문장)",
  "decisions": ["결정사항 1", "결정사항 2"],
  "action_items": [
    {{"owner": "담당자", "task": "작업 내용", "deadline": "마감일 (없으면 null)"}}
  ],
  "next_meeting": "다음 회의 날짜 (없으면 null)"
}}

회의 메모:
{MEETING_MEMO}"""

mock_json = {
    "summary": "CloudSync v2.3 배포를 11/26으로 확정하고, 마이그레이션 스크립트 검토와 버그 수정 등 사전 준비 사항을 점검한 스프린트 회의.",
    "decisions": [
        "v2.3 배포일: 2024-11-26 (화)",
        "롤백 기준: 에러율 5% 초과 시",
        "다음 스프린트 플래닝: 2024-12-02"
    ],
    "action_items": [
        {"owner": "이수진", "task": "마이그레이션 스크립트 최종 검토", "deadline": "2024-11-22 (금)"},
        {"owner": "박민재", "task": "로딩 스피너 버그 수정", "deadline": "2024-11-20 (오늘)"},
        {"owner": "최서연", "task": "QA 발견 이슈 3개 Jira 등록", "deadline": None},
        {"owner": "김도현", "task": "보안팀 이민준에게 API 키 감사 결과 연락", "deadline": None},
        {"owner": "이수진", "task": "신규 ML 엔지니어 온보딩 멘토 역할", "deadline": None}
    ],
    "next_meeting": "2024-12-02"
}

result_b = call_llm_json(prompt_json, mock=mock_json)

print("=" * 55)
print("  패턴 B: 구조화 JSON 출력")
print("=" * 55)
print(f"\n📋 요약: {result_b.get('summary', '')}")
print(f"\n✅ 결정 사항:")
for d in result_b.get("decisions", []):
    print(f"   • {d}")

print(f"\n📌 액션 아이템:")
items = result_b.get("action_items", [])
df_items = pd.DataFrame(items)
if not df_items.empty:
    df_items.columns = ["담당자", "작업", "마감일"]
    display(df_items)

print(f"\n📅 다음 회의: {result_b.get('next_meeting', '미정')}")
print()
print("✅ 이 JSON은 Jira API, Slack Bot, DB 저장 등에 바로 연결할 수 있습니다.")

## 5️⃣ 패턴 C: 함수 인자 파싱 (자연어 → 구조)

고객의 자연어 문의를 **Support Ticket JSON**으로 변환합니다.  
이 패턴이 **Function Calling / Tool Use**의 기반이 됩니다.

In [ ]:
# ─────────────────────────────────────────
# 패턴 C-1: 고객 문의 → Support Ticket JSON
# ─────────────────────────────────────────
prompt_ticket = f"""고객 문의 텍스트를 Support Ticket JSON으로 변환하세요.

반환 형식:
{{
  "priority": "urgent|high|normal|low",
  "category": "bug|feature_request|question|billing",
  "product": "제품명",
  "version": "관련 버전 (없으면 null)",
  "error_message": "에러 메시지 (없으면 null)",
  "summary": "문제 요약 (1문장)",
  "suggested_kb_tags": ["검색에 쓸 키워드 3개"]
}}

고객 문의:
{SUPPORT_TICKET_TEXT}"""

mock_ticket = {
    "priority": "urgent",
    "category": "bug",
    "product": "CloudSync",
    "version": "v2.3",
    "error_message": "Authentication failed: JWT token not supported",
    "summary": "CloudSync v2.2→v2.3 업그레이드 후 JWT 인증 실패로 Python 스크립트 동작 불가",
    "suggested_kb_tags": ["CloudSync v2.3", "JWT OAuth 마이그레이션", "Python 3.9 호환성"]
}

result_c1 = call_llm_json(prompt_ticket, mock=mock_ticket)

print("=" * 55)
print("  패턴 C-1: 고객 문의 → Support Ticket")
print("=" * 55)
print(f"\n원본 텍스트:")
print(f"  '{SUPPORT_TICKET_TEXT[:80]}...'")
print(f"\n변환 결과:")
for k, v in result_c1.items():
    print(f"  {k:25s}: {v}")

In [ ]:
# ─────────────────────────────────────────
# 패턴 C-2: 자연어 검색 요청 → 구조화된 검색 파라미터
# (Function Calling 시뮬레이션)
# ─────────────────────────────────────────

SEARCH_FUNCTION_SPEC = """
사용 가능한 함수:
- search_docs(query: str, category: str, top_k: int): 문서 검색
- lookup_release_note(product: str, version: str): 릴리즈 노트 조회
- lookup_faq(product: str, keywords: list): FAQ 검색
"""

prompt_func = f"""사용자의 자연어 요청을 분석하여 어떤 함수를 어떤 인자로 호출해야 할지 JSON으로 반환하세요.

{SEARCH_FUNCTION_SPEC}

반환 형식:
{{
  "function_calls": [
    {{"function": "함수명", "arguments": {{"인자명": "값"}}, "reason": "왜 이 함수를 쓰는지"}}
  ]
}}

사용자 요청:
{USER_SEARCH_REQUEST}"""

mock_func = {
    "function_calls": [
        {
            "function": "lookup_release_note",
            "arguments": {"product": "DataPulse", "version": "latest"},
            "reason": "'지난 달에 배포된 최신 버전'을 알기 위해 릴리즈 노트 조회 필요"
        },
        {
            "function": "search_docs",
            "arguments": {"query": "DataPulse Kafka 연동 설정", "category": "릴리즈노트", "top_k": 3},
            "reason": "Kafka 연동 설정 방법 검색"
        },
        {
            "function": "lookup_faq",
            "arguments": {"product": "DataPulse", "keywords": ["Slack", "알림", "설정"]},
            "reason": "Slack 알림 설정은 FAQ에서 찾을 가능성이 높음"
        }
    ]
}

result_c2 = call_llm_json(prompt_func, mock=mock_func)

print("=" * 55)
print("  패턴 C-2: 자연어 요청 → 함수 호출 계획")
print("=" * 55)
print(f"\n원본 요청:")
print(f"  '{USER_SEARCH_REQUEST[:80]}...'")
print(f"\nLLM이 결정한 함수 호출 계획:")
for i, call in enumerate(result_c2.get("function_calls", []), 1):
    print(f"\n  [{i}] {call['function']}({call['arguments']})")
    print(f"       이유: {call['reason']}")

print()
print("💡 이것이 Tool Use / Function Calling의 핵심 패턴입니다.")
print("   06번 노트북에서 이 패턴으로 실제 Agentic Search를 구현합니다.")

## 6️⃣ 핵심 비교: 프롬프트 품질이 곧 결과 품질

In [ ]:
# ─────────────────────────────────────────
# 같은 질문, 다른 프롬프트 → 품질 차이 시연
# ─────────────────────────────────────────

question = "보안 정책이 뭔가요?"

# 프롬프트 A: 너무 단순
prompt_bad = question

# 프롬프트 B: 역할 + 제약 + 형식
prompt_good = f"""당신은 테크코어 내부 지식 어시스턴트입니다.
질문에 대해 아래 형식으로 답변해주세요:

형식:
- 핵심 요점 (최대 3개, 불릿 형식)
- 담당 부서 또는 연락처
- 관련 문서 ID

만약 정보가 불확실하면 '확인이 필요합니다'라고 명시하세요.

질문: {question}"""

mock_bad = "보안 정책은 회사의 정보 보안을 위한 규칙들입니다. API 키 관리, 접근 권한 등이 포함됩니다."

mock_good = """핵심 요점:
• API 키는 개발용 90일, 운영용 365일 유효 (SEC-POL-003 v3.1 기준)
• 코드베이스 하드코딩 금지 - 환경변수 또는 AWS Secrets Manager 필수
• 키 유출 시 10분 이내 폐기 의무, Slack #security-alert 즉시 보고

담당 부서: 보안팀 이민준 (minjun.lee@techcore.kr)
관련 문서: SEC-POL-003 (현행), SEC-POL-002 (구버전 - 비효력)"""

resp_bad = call_llm(prompt_bad, mock=mock_bad)
resp_good = call_llm(prompt_good, mock=mock_good)

# HTML로 나란히 비교
bad_html = resp_bad.replace('\n', '<br>')
good_html = resp_good.replace('\n', '<br>')

display(HTML(f"""
<div style="font-family:Arial,sans-serif;max-width:900px;margin:10px auto;">
  <div style="background:#2c3e50;color:white;padding:10px 15px;border-radius:8px 8px 0 0;">
    <strong>🔍 질문:</strong> {question}
  </div>
  <div style="display:grid;grid-template-columns:1fr 1fr;border:1px solid #ddd;border-top:none;">
    <div style="padding:15px;background:#fdf5f5;border-right:1px solid #ddd;">
      <div style="font-weight:bold;color:#c0392b;margin-bottom:10px;">❌ 단순 프롬프트</div>
      <code style="font-size:11px;color:#888;">prompt: '{question}'</code>
      <hr style="border-color:#eee;">
      <div style="font-size:13px;line-height:1.7;">{bad_html}</div>
    </div>
    <div style="padding:15px;background:#f0fdf4;">
      <div style="font-weight:bold;color:#27ae60;margin-bottom:10px;">✅ 구조화된 프롬프트</div>
      <code style="font-size:11px;color:#888;">역할 + 제약 + 형식 지정</code>
      <hr style="border-color:#eee;">
      <div style="font-size:13px;line-height:1.7;">{good_html}</div>
    </div>
  </div>
  <div style="background:#fff9c4;padding:10px 15px;border:1px solid #ddd;border-top:none;font-size:13px;">
    💡 <strong>같은 LLM, 같은 질문, 다른 프롬프트</strong> → 완전히 다른 결과 품질
  </div>
</div>
"""))

## 7️⃣ 실제 Function Calling (API 모드에서만)

OpenAI의 공식 Function Calling 기능을 보여줍니다.

In [ ]:
# OpenAI Function Calling 예시 (API 모드에서만 실행)

if MODE == "api" and client:
    # 도구 정의
    tools = [
        {
            "type": "function",
            "function": {
                "name": "search_docs",
                "description": "내부 문서를 검색합니다",
                "parameters": {
                    "type": "object",
                    "properties": {
                        "query": {"type": "string", "description": "검색 쿼리"},
                        "category": {
                            "type": "string",
                            "enum": ["보안정책", "릴리즈노트", "FAQ", "회의록", "프로세스"],
                            "description": "문서 카테고리"
                        },
                        "top_k": {"type": "integer", "description": "반환할 문서 수"}
                    },
                    "required": ["query"]
                }
            }
        }
    ]

    try:
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": USER_SEARCH_REQUEST}],
            tools=tools,
            tool_choice="auto"
        )

        msg = response.choices[0].message
        print("=" * 55)
        print("  실제 Function Calling 결과")
        print("=" * 55)

        if msg.tool_calls:
            for tc in msg.tool_calls:
                args = json.loads(tc.function.arguments)
                print(f"\n🔧 함수 호출: {tc.function.name}")
                for k, v in args.items():
                    print(f"   {k}: {v}")
        else:
            print(msg.content)

    except Exception as e:
        print(f"오류: {e}")
else:
    # Mock 시연
    print("[Mock - API 모드에서 실행 시 실제 Function Calling 결과]")
    print()
    print("요청:", USER_SEARCH_REQUEST[:60], "...")
    print()
    print("LLM이 자동으로 선택한 함수:")
    print("  1. lookup_release_note(product='DataPulse', version='latest')")
    print("  2. search_docs(query='Kafka 연동 설정', category='릴리즈노트')")
    print("  3. lookup_faq(product='DataPulse', keywords=['Slack', '알림'])")
    print()
    print("💡 LLM이 자연어를 분석해서 어떤 함수를 호출할지 스스로 결정합니다.")

---

## 8️⃣ 한계와 주의점

In [ ]:
# LLM 구조화의 한계 - 직접 확인해보기

print("⚠️  LLM 인터페이스의 한계점")
print("─" * 50)
print()
print("1. 할루시네이션 (Hallucination)")
print("   → LLM은 없는 정보를 그럴듯하게 지어낼 수 있습니다.")
print("   → 해결: RAG로 실제 문서를 기반으로 답변하게 만들기")
print()
print("2. 컨텍스트 길이 제한")
print("   → 너무 긴 문서는 프롬프트에 통째로 넣을 수 없습니다.")
print("   → 해결: 청킹 + 검색으로 관련 부분만 주입")
print()
print("3. 학습 데이터 컷오프")
print("   → LLM은 학습 이후의 정보를 모릅니다.")
print("   → 해결: RAG, Tool Use, 실시간 데이터 연결")
print()
print("4. 비결정적 출력")
print("   → 같은 프롬프트도 실행마다 다를 수 있습니다.")
print("   → 해결: temperature=0, JSON 형식 강제, 테스트")
print()
print("→ 이 한계들을 해결하는 과정이 03~06 노트북의 주제입니다.")

---

## 🎤 강의자 멘트 포인트

> **"여러분이 LLM을 쓸 때 가장 중요한 질문은 '어떤 모델을 쓸까'가 아니라  
> '어떤 출력 구조를 요구할까'입니다.**  
>
> 패턴 A처럼 자유 생성을 쓰면 '말은 잘하는 챗봇'이 됩니다.  
> 패턴 B, C처럼 구조를 요구하면 LLM은 **시스템의 인터페이스**가 됩니다.  
> 회의 메모가 자동으로 Jira 티켓이 되고,  
> 고객 문의가 자동으로 검색 쿼리가 되는 세상입니다."

## 🙋 청중 질문 유도
> - "여러분 업무에서 LLM을 쓴다면 어떤 '구조화'가 도움이 될까요?"
> - "JSON 출력이 항상 완벽할까요? 어떤 경우에 실패할 수 있을까요?"
> - "Function Calling을 쓰면 어떤 새로운 가능성이 열릴까요?"

## 🏗️ 실무 확장 포인트

이 데모를 실제 서비스로 확장하면:
- **Prompt versioning**: 프롬프트도 코드처럼 버전 관리 필요
- **Output validation**: JSON 스키마 검증 (Pydantic 등)
- **Evaluation**: LLM 출력 품질을 자동 평가하는 파이프라인
- **Logging**: 모든 LLM 호출을 추적 및 모니터링
- **Caching**: 동일 입력에 대한 응답 캐싱으로 비용 절감
- **Rate limiting & retry**: API 실패 대응

## ➕ 추가 실험 아이디어
1. `temperature`를 0.0 vs 1.0으로 바꿔서 같은 질문을 5번씩 실행해보기
2. 회의 메모를 영어로 바꿔서 JSON 추출 품질 비교
3. JSON 스키마에 없는 필드를 요청하면 어떻게 되는지 확인
4. 고의로 잘못된 JSON 출력을 유도하는 프롬프트 만들어보기

## ➡️ 다음 노트북
**02_multimodal_demo.ipynb** - 텍스트만이 아니라 이미지/문서까지 처리하는 멀티모달